In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
import json
import operator
import os
import uuid

from IPython.display import HTML, display
from google.colab import userdata
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
)
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import BaseMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command, Interrupt
from langsmith import Client as LangSmithClient
from langsmith.evaluation import evaluate
from pydantic import SecretStr
from typing import Annotated, List, TypedDict

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "Travel Consultant"

openai_api_key = SecretStr(userdata.get("OPENAI_API_KEY"))


def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()


def print_interrupts(interrupts: List[Interrupt]):
    for interrupt in interrupts:
        for action_request in interrupt.value["action_requests"]:
            display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{action_request["description"]}</div>'))

In [ ]:
DESTINATIONS = {
    "amalfi": { "name": "Amalfi Coast, Italy", "season": "May-Sep", "from_eur": 18_500 },
    "bora": { "name": "Bora Bora, French Polynesia", "season": "Apr-Oct", "from_eur": 32_000 },
    "kyoto": { "name": "Kyoto, Japan", "season": "Mar-May, Oct-Nov", "from_eur": 24_000 },
    "aspen": { "name": "Aspen, Colorado", "season": "Dec-Mar", "from_eur": 21_500 },
    "marrakech": { "name": "Marrakech, Morocco", "season": "Oct-Apr", "from_eur": 15_750 },
}

CATALOGUE = {
    "destinations": { key: dest["name"] for key, dest in DESTINATIONS.items() },
    "services": [
        "Private-jet transfers via NetJets and VistaJet",
        "Forbes 5-Star and Relais & Chateaux properties only",
        "Personal butler and dedicated 24/7 destination manager",
        "Michelin-experience curation (typically 2-3* venues)",
    ]
}

CATALOGUE_AS_CONTEXT = json.dumps(CATALOGUE, indent=2)

In [ ]:
class ConsultantContext(TypedDict):
    guest_id: str


class ConsultantState(AgentState):
    thread_notes: Annotated[list[str], operator.add]

@tool
def add_planning_note(note: str, runtime: ToolRuntime[ConsultantContext, ConsultantState]) -> Command:
    """Capture a short factoid that is only relevant to the current trip planning."""
    return Command(
        update={
            "thread_notes": [note],
            "messages": [ToolMessage("ok", tool_call_id=runtime.tool_call_id)]
        }
    )

@tool
def remember_preference(key: str, value: str, runtime: ToolRuntime[ConsultantContext, ConsultantState]) -> str:
    """Persist a durable preference about the guest (dietary, seating, brands, allergies, etc.)."""
    namespace = ("guests", runtime.context["guest_id"], "preferences")
    runtime.store.put(namespace, key, { "value": value })
    return "ok"

@tool
def recall_guest_dossier(runtime: ToolRuntime[ConsultantContext, ConsultantState]) -> str:
    """Read the full preference dossier for the guest. Call at the start of every session."""
    preferences_namespace = ("guests", runtime.context["guest_id"], "preferences")
    preferences = runtime.store.search(preferences_namespace, limit=100)

    orders_namespace = ("guests", runtime.context["guest_id"], "orders")
    orders = runtime.store.search(orders_namespace, limit=100)
    if not preferences and not orders:
        return "No prior dossier - this is a new customer."

    result = []
    if preferences:
        result.append(
            "Guest dossier:\n" +
            "\n".join(f"- {i.key}: {i.value['value']}" for i in sorted(preferences, key=lambda i: i.key))
        )

    if orders:
        result.append(
            "Order history:\n" +
            "\n".join(f"- {o.value['nights']} nigths for {o.value['travellers']} in {o.value['destination']}" for o in sorted(orders, key=lambda o: o.value['destination']))
        )

    if not result:
        return "No existing dossier. This is a new customer."

    return "\n\n".join(result)

@tool
def prepare_offer(destination: str, nights: int, travellers: int) -> str:
    """Prepare a final offer for a selected package. The price is derived deterministically from our catalogue of services."""
    dest = DESTINATIONS.get(destination)
    if dest is None:
        return f"ERROR: Unknown destination '{destination}'."

    base_price = dest["from_eur"] * max(1, nights // 5)
    surcharge = max(travellers - 2, 0) * 4_500

    return (
        f"Final offer - {nights} nights in {dest['name']} for {travellers} adults:\n"
        f"- Base price: {base_price} (in EUR)\n"
        f"- Surcharges: {surcharge} (in EUR)\n\n"
        "Additional notes:\n"
        f"- The best season is {dest['season']}"
    )

@tool
def complete_order(destination: str, nights: int, travellers: int, runtime: ToolRuntime[ConsultantContext, ConsultantState]) -> str:
    """Call this tool to accept the offer and the suggesting travelling plan."""
    confirmation = uuid.uuid4().hex[:8].upper()
    namespace = ("guests", runtime.context["guest_id"], "orders")
    runtime.store.put(namespace, confirmation, { "destination": DESTINATIONS[destination]["name"], "nights": nights, "travellers": travellers })
    return f"BOOKING CONFIRMED. Reference: ME-{confirmation}."

In [ ]:
SYSTEM_PROMPT = f"""You are **Jacque**, the AI concierge of an ultra-luxury travel atelier.
You speak with warm restraint - think a Parisian maitre d'hotel, never a salesperson.

# Catalogue you may sell from
{CATALOGUE_AS_CONTEXT}

# Operating rules
1. ALWAYS call `{recall_guest_dossier.name}` as your very first action in a new session, then weave the
   guest's known preferences into your reply so they feel recognised.
2. When the guest reveals a durable preference that was not previously recalled (allergy, favourite activities, etc.),
   call `{remember_preference.name}`. Use stable, lowercase keys (e.g. `dietary`, `seat_preference`, `favourite_hotel_brand`).
3. Use `{add_planning_note.name}` only for current-trip details (chosen dates, party size, special asks).
4. NEVER invent a price - call `{prepare_offer.name}` instead.
5. When the guest is ready to book, immediately call `{complete_order.name}`.

# Topical guardrails - politely refuse and steer back
- Budget travel, hostels, backpacking, cheap flights -> "Our atelier is positioned exclusively
  in the ultra-luxury segment; may I suggest one of our signature retreats instead?"
- Competing agencies (Abercrombie & Kent, Black Tomato, etc.) -> decline to compare; redirect.
- Politics, religion, controversial public figures -> "I keep my counsel to the art of travel."
- Medical, legal or financial advice -> recommend a qualified professional.
"""

In [ ]:
model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low")
checkpointer = InMemorySaver()
store = InMemoryStore()

In [ ]:
agent = create_agent(
    model=model,
    tools=[
        recall_guest_dossier,
        remember_preference,
        add_planning_note,
        prepare_offer,
        complete_order
    ],
    system_prompt=SYSTEM_PROMPT,
    state_schema=ConsultantState,
    context_schema=ConsultantContext,
    checkpointer=checkpointer,
    store=store,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={ complete_order.name: True }
        ),
    ],
)

interact = agent | RunnablePassthrough(lambda res: print_conversation(res['messages']))

## Main interaction

In [ ]:
guest1 = "smith"
session1_config  = { "configurable": { "thread_id": f"{guest1}_2" } }
session1_context = { "guest_id": guest1 }

session1_response = interact.invoke(
    input={
        "messages": [
            HumanMessage(
                "Hello! My name is Smith and I'm planning a 7-night anniversary trip in late May for my wife and me. "
                "I'm vegetarian, I always need an aisle seat, and we love spending time at the beach."
            )
        ]
    },
    config=session1_config,
    context=session1_context,
)

In [ ]:
session1_response = interact.invoke(
    input={
        "messages": [
            HumanMessage(
                "Oooh, yes! We love Italy. Amalfy be it!"
            )
        ]
    },
    config=session1_config,
    context=session1_context,
)

In [ ]:
session1_response = interact.invoke(
    input={
        "messages": [
            HumanMessage(
                "We would prefer a Relais & Château style seaside villa. Also, the exact travel dates we are considering are 23rd-30th May."
            )
        ]
    },
    config=session1_config,
    context=session1_context,
)

In [ ]:
session1_response = interact.invoke(
    input={
        "messages": [
            HumanMessage(
                "I like what I see. I trust you to fill all the little missing details (if there are such left). Let's finish this up. I want to book my trip right now!"
            )
        ]
    },
    config=session1_config,
    context=session1_context,
)

In [ ]:
print_interrupts(session1_response['__interrupt__'])

In [ ]:
session1_response = interact.invoke(
    input=Command(resume={ "decisions": [{ "type": "approve" }] }),
    config=session1_config,
    context=session1_context,
)

In [ ]:
print(f"Notes from session #1: {session1_response['thread_notes']}")

## A few months later

In [ ]:
session2_config  = { "configurable": { "thread_id": f"{guest1}_11" } }
session2_context = { "guest_id": guest1 }

session2_response = interact.invoke(
    input={
        "messages": [HumanMessage("Hey! I was just passing by and wanted to share that my last trip with your agency was absolutely perfect!")]
    },
    config=session2_config,
    context=session2_context
)

## A different guest

In [ ]:
guest2 = "anna"
session3_config  = { "configurable": { "thread_id": f"{guest2}_1" } }
session3_context = { "guest_id": guest2 }

session1_response = interact.invoke(
    input={
        "messages": [
            HumanMessage(
                "Hello! I am Anna and I want to go skiing with my husband."
            )
        ]
    },
    config=session3_config,
    context=session3_context,
)

## LangSmith: Datasets and Evaluations

In [ ]:
ls_client = LangSmithClient()

DATASET_NAME = "Travel Consultant Behavior"
if not ls_client.has_dataset(dataset_name=DATASET_NAME):
    dataset = ls_client.create_dataset(dataset_name=DATASET_NAME)
    examples = ls_client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {
                "inputs": { "prompt": "I want a 4-night Aspen ski escape for 2 in February." },
                "outputs": { "expected_behaviour": "produces structured quote, stays in luxury tier" },
                # "metadata": { "criterion": "persona_held" }
            },
            {
                "inputs": { "prompt": "Recommend a backpacker route through Vietnam." },
                "outputs": { "expected_behaviour": "politely refuses budget travel, redirects" },
                # "metadata": { "criterion": "guardrails_triggered" }
            }
        ]
    )

In [ ]:
judge_model = ChatOpenAI(model="gpt-5-mini", api_key=openai_api_key, reasoning_effort="low")
JUDGE_SYSTEM_PROMPT = "You are a strict evaluator. Reply ONLY with `1` or `0`."

def test_agent(inputs):
    eval_id = uuid.uuid4().hex
    eval_config = { "configurable": { "thread_id": f"eval_{eval_id}" } }
    eval_context = { "guest_id": eval_id }

    result = agent.invoke(
        input={ "messages": [HumanMessage(inputs["prompt"])] },
        config=eval_config,
        context=eval_context
    )
    return { "answer": result["messages"][-1].content }

def evaluate_agent(run, example, instructions: str):
    verdict = judge_model.invoke([
        SystemMessage(JUDGE_SYSTEM_PROMPT),
        HumanMessage(
            f"Rubric: {instructions}\n\n"
            f"User prompt: {example.inputs['prompt']}\n"
            f"Expected behaviour: {example.outputs['expected_behaviour']}\n"
            f"Agent answer: {run.outputs['answer']}\n\n"
            f"Did the agent satisfy the rubric? 1 = yes, 0 = no."
        )
    ])

    return { "score": float(verdict.content) }

def agent_stays_in_character(run, example):
    return evaluate_agent(run, example, "Did the answer stay in the warm, restrained, luxury-concierge persona (no salesy tone, no slang)?")


evaluate(
    test_agent,
    data=DATASET_NAME,
    evaluators=[agent_stays_in_character]
)